# Polarized Light and Geometric Phase in qg Units: Quantum Optics & QPON Networks

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Viny2030/qang/blob/main/notebooks/qang_optics_en.ipynb)

This notebook implements the systematic unification of Stokes polarimetry, lossless Mueller transformations, and geometric Berry phase within the $qg$ framework, as developed in the 35-chapter treatise *Computación, información y física cuántica en el espacio qang* (Part VII and Quantum Passive Optical Networks - QPON).

### Key Formulation Principles (qang v0.14.0):
1. **Normalized Stokes parameters are $qg$ values:** The Poincaré sphere matches the Bloch sphere. With $|H\rangle = |0\rangle$ and $|V\rangle = |1\rangle$:
   * $qg_Z = S_1 / S_0$ ($H - V$)
   * $qg_X = S_2 / S_0$ ($D - A$)
   * $qg_Y = S_3 / S_0$ ($R - L$)
2. **Lossless Mueller matrices map to the $qg$ gate rule:** $qg'_P = qg_{U^\dagger P U}$.
3. **Malus's Law & Photon Counting Polarimetry:** Formulated in $mqg$ (milli-qang) resolution with rigorous confidence intervals.
4. **Berry & Pancharatnam Geometric Phase:** Evaluated as Stokes' theorem on the unit sphere: $\gamma = -\Omega / 2$, i.e., **$(qg_Z - 1)/2$ turns**.

## 1. Setup and Module Loading (`qang 0.14.0`)

In [ ]:
import importlib, subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "qang>=0.14.0", "matplotlib", "numpy"], check=False)
importlib.invalidate_caches()

import numpy as np
import matplotlib.pyplot as plt
import qang
from qang import formulation as F
from qang import geometric as G
from qang import polarization as P

print(f"qang framework loaded: v{qang.__version__}")

## 2. Stokes Parameters are Exact qg Values

In [ ]:
s2 = np.sqrt(2)
states = {
    "H": [1, 0], 
    "V": [0, 1], 
    "D": [1 / s2, 1 / s2], 
    "A": [1 / s2, -1 / s2],
    "R": [1 / s2, 1j / s2], 
    "L": [1 / s2, -1j / s2], 
    "Elliptic": [0.6, 0.8 * np.exp(0.7j)]
}

print(f"{'State':<12}{'with qang (Stokes -> qg)':>35}{'Standard (<P>)':>35}{'Difference':>14}")
fmt = lambda d: "(" + ", ".join(f"{d[k]:+.3f}" for k in "XYZ") + ")"
for name, E in states.items():
    q = P.jones_to_qg(E)
    ref = F.qg_values(np.array(E, dtype=complex))
    ref = {k: ref.get(k, 0.0) for k in "XYZ"}
    diff = max(abs(q[k] - ref[k]) for k in "XYZ")
    print(f"{name:<12}{fmt(q):>35}{fmt(ref):>35}{diff:>14.1e}")

## 3. Partially Polarized Light & Mixed States

In [ ]:
E = np.array([0.6, 0.8 * np.exp(0.7j)])
print("Depolarization | Degree Pol. P | Purity (qang) | Tr(rho^2)")
for p in (0.0, 0.3, 0.6, 0.9):
    S = P.depolarizer(1 - p) @ P.jones_to_stokes(E)
    q = P.stokes_to_qg(S)
    rho = F.state_from_qg({"I": 1.0, **q}, 1)
    print(f"     {p:.1f}       |    {P.degree_of_polarization(S):.3f}      |     {P.purity_from_stokes(S):.4f}    |   {np.real(np.trace(rho @ rho)):.4f}")

## 4. Lossless Mueller Matrix is the qg Gate Transformation Rule

In [ ]:
rng = np.random.default_rng(42)
components = {
    "Half-wave 0.3 rad": P.half_wave_plate(0.3),
    "Quarter-wave 1.1 rad": P.quarter_wave_plate(1.1),
    "Rotator 0.7 rad": P.rotator(0.7)
}

for name, J in components.items():
    err = 0.0
    for _ in range(10):
        E = rng.normal(size=2) + 1j * rng.normal(size=2)
        E /= np.linalg.norm(E)
        q_in = P.jones_to_qg(E)
        q_mueller = P.stokes_to_qg(P.mueller_from_jones(J) @ P.qg_to_stokes(q_in))
        q_rule = F.apply_gate({"I": 1.0, **q_in}, J)
        err = max(err, max(abs(q_mueller[k] - q_rule.get(k, 0.0)) for k in "XYZ"))
    print(f"{name:<24} max discrepancy Mueller vs qg rule: {err:.1e}")

## 5. Malus's Law in qg Coordinates

In [ ]:
theta = np.linspace(0, np.pi, 200)
fig, ax = plt.subplots(figsize=(6.5, 3.4))

plot_states = [("H", [1, 0], "#2a78d6"), ("D", [1/s2, 1/s2], "#eb6834"), 
               ("R", [1/s2, 1j/s2], "#1baf7a"), ("Elliptic", [0.6, 0.8 * np.exp(0.7j)], "#5f5e58")]

for name, E, color in plot_states:
    q = P.jones_to_qg(E)
    ax.plot(theta, [P.malus_intensity(q, t) for t in theta], color=color, lw=1.8, label=name)
    pts = np.linspace(0, np.pi, 9)
    ax.plot(pts, [(P.mueller_from_jones(P.linear_polarizer(t)) @ P.qg_to_stokes(q))[0] for t in pts], "o", color=color, ms=4)

ax.set_xlabel("Polarizer angle θ (rad)")
ax.set_ylabel("Normalized intensity I/I₀")
ax.set_title("Malus's Law in qg Units (Solid: qg formula, Dots: Mueller)")
ax.legend(frameon=False, ncol=4, fontsize=8, loc="upper center")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Geometric Phase and Stokes' Theorem

In [ ]:
def cone(qg_z, n=2000):
    s = np.sqrt(1 - qg_z**2)
    t = np.linspace(0, 2 * np.pi, n, endpoint=False)
    return np.stack([s * np.cos(t), s * np.sin(t), np.full(n, qg_z)], axis=1)

zs = np.linspace(-0.9, 0.9, 7)
over = [G.geometric_phase_qg(cone(z)).phi for z in zs]
solid = [(G.berry_phase_from_solid_angle(G.solid_angle(cone(z), reference=[0, 0, 1])) / (2 * np.pi)) % 1 for z in zs]
flux = [(-G.curvature_flux(z) / (2 * np.pi)) % 1 for z in zs]

zz = np.linspace(-1, 0.995, 200)
fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.plot(zz, ((zz - 1) / 2) % 1, color="#5f5e58", lw=2, label="(qg_Z - 1)/2 mod 1")
ax.plot(zs, over, "o", color="#2a78d6", ms=7, label="State overlaps (Pancharatnam)")
ax.plot(zs, solid, "s", color="#eb6834", ms=5, label="Solid angle")
ax.plot(zs, flux, "^", color="#1baf7a", ms=4, label="Curvature flux (Stokes)")

ax.set_xlabel("qg_Z of the cone loop")
ax.set_ylabel("Geometric phase (turns)")
ax.set_title("Stokes' Theorem and Berry Phase in qg Space")
ax.legend(frameon=False, fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

max_diff = max(abs(((a - (z - 1) / 2) + 0.5) % 1 - 0.5) for a, z in zip(over, zs))
print(f"Max difference from (qg_Z - 1)/2: {max_diff:.2e}")

## Conclusions
* **Polarization-Quantum Mapping:** Normalized Stokes coordinates rigorously coincide with the $qg$ unit vector on the Poincaré/Bloch sphere.
* **Optical Network Verification:** Lossless components obey quantum gate transformation rules $qg' = U^\dagger P U$, providing a unified diagnostic language for photonic qubits and quantum optical networks.